# Solana DEX Analysis Notebook

Pump.fun + Raydium transaction forensics and ML inference results.

**Data sources:**
- `feed/live_features.jsonl` — raw transactions with 32-dim features
- `feed/bots.jsonl` — bot forensics records
- `feed/bot_summary.json` — aggregate stats
- `models/training.log` — model training metrics

In [ ]:
import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np

# Matplotlib with dark theme
import matplotlib.pyplot as plt
plt.style.use('dark_background')

FEED = Path('./feed/live_features.jsonl')
BOTS = Path('./feed/bots.jsonl')
SUMMARY = Path('./feed/bot_summary.json')
TRAIN_LOG = Path('./models/training.log')

print('Data sources:')
for p in [FEED, BOTS, SUMMARY, TRAIN_LOG]:
    size = p.stat().st_size / (1024**2) if p.exists() else 0
    print(f"  {p}  {size:.1f} MB")

## 1. Load Transaction Feed

In [ ]:
def load_jsonl(path, max_lines=100000):
    rows = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows

txs = load_jsonl(FEED)
print(f"Loaded {len(txs)} transactions")

# Quick peek
df = pd.DataFrame(txs)
df.head()

## 2. Feature Distribution

In [ ]:
from pump_constants import FEATURE_NAMES

# Extract feature matrix
feats = np.array([t['features'] for t in txs if len(t.get('features', [])) == 32])
print(f"Feature matrix shape: {feats.shape}")

fig, axes = plt.subplots(4, 8, figsize=(24, 12))
axes = axes.flatten()
for i, name in enumerate(FEATURE_NAMES):
    ax = axes[i]
    data = feats[:, i]
    # Clip outliers for viz
    q99 = np.percentile(data[data != 0], 99) if np.any(data != 0) else 1
    data_clipped = np.clip(data, -q99, q99)
    ax.hist(data_clipped[data_clipped != 0], bins=50, color='cyan', alpha=0.7)
    ax.set_title(name, fontsize=8)
    ax.tick_params(labelsize=6)
plt.tight_layout()
plt.show()

## 3. Bot Forensics Results

In [ ]:
bots = load_jsonl(BOTS)
print(f"Bot records: {len(bots)}")

if bots:
    bot_df = pd.DataFrame(bots)
    
    # Tag distribution
    all_tags = [tag for b in bots for tag in b.get('tags', [])]
    tag_counts = Counter(all_tags)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Tag bar chart
    ax1 = axes[0]
    tags, counts = zip(*tag_counts.most_common())
    ax1.barh(tags, counts, color='crimson')
    ax1.set_xlabel('Count')
    ax1.set_title('Bot Tag Distribution')
    
    # Bot score histogram
    ax2 = axes[1]
    scores = [b['bot_score'] for b in bots if 'bot_score' in b]
    ax2.hist(scores, bins=30, color='orange', alpha=0.7, edgecolor='white')
    ax2.set_xlabel('Bot Score')
    ax2.set_ylabel('Count')
    ax2.set_title('Bot Score Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Top bots table
    print("\nTop 10 bot wallets:")
    bot_df.nlargest(10, 'bot_score')[['wallet', 'bot_score', 'tags', 'slot']]

## 4. Training Metrics

In [ ]:
if TRAIN_LOG.exists():
    train_rows = []
    with open(TRAIN_LOG) as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) == 4:
                train_rows.append({
                    'ts': float(parts[0]),
                    'step': int(parts[1]),
                    'loss': float(parts[2]),
                    'samples': int(parts[3]),
                })
    train_df = pd.DataFrame(train_rows)
    print(f"Training steps: {len(train_df)}")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1 = axes[0]
    ax1.plot(train_df['step'], train_df['loss'], color='lime', linewidth=0.8)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.set_yscale('log')
    
    ax2 = axes[1]
    ax2.plot(train_df['step'], train_df['samples'], color='cyan', linewidth=0.8)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Samples')
    ax2.set_title('Dataset Growth')
    
    plt.tight_layout()
    plt.show()

## 5. Wallet Clustering (Bot vs Human)

In [ ]:
from collections import defaultdict

wallet_stats = defaultdict(lambda: {'tx_count': 0, 'total_sol': 0, 'tags': set()})
for b in bots:
    w = b.get('wallet', '')
    wallet_stats[w]['tx_count'] = b.get('wallet_stats', {}).get('tx_count', 0)
    wallet_stats[w]['total_sol'] += b.get('sol_delta', 0)
    wallet_stats[w]['tags'].update(b.get('tags', []))

if wallet_stats:
    wdf = pd.DataFrame([
        {'wallet': k, **v, 'tag_count': len(v['tags'])}
        for k, v in wallet_stats.items()
    ])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(
        wdf['tx_count'],
        wdf['total_sol'],
        c=wdf['tag_count'],
        cmap='YlOrRd',
        s=60,
        alpha=0.7,
        edgecolors='white',
        linewidth=0.3
    )
    ax.set_xlabel('Transaction Count')
    ax.set_ylabel('Total SOL Delta')
    ax.set_title('Wallet Clustering: Tx Count vs SOL Delta (color = bot tags)')
    ax.set_xscale('log')
    ax.set_yscale('symlog')
    plt.colorbar(scatter, ax=ax, label='Bot Tags')
    plt.show()

## 6. Real-Time Inference Alerts (Simulated)

In [ ]:
# Simulate inference by scoring recent transactions with a heuristic
# (In production, gpu_trainer.py infer_worker() does this on GPU 0)

def heuristic_score(feats):
    """Simple heuristic: high price velocity + buy ratio = pump signal."""
    if len(feats) < 32:
        return 0.0
    price_vel = feats[15]
    buy_ratio = feats[13]
    bot_score = feats[20]
    return min(1.0, price_vel * 10 + buy_ratio * 0.3 - bot_score * 0.2)

recent = txs[-1000:]
scores = [(t.get('signature', '')[:12], heuristic_score(t.get('features', []))) for t in recent]
scores.sort(key=lambda x: x[1], reverse=True)

print("Top 10 pump signals (heuristic):")
for sig, score in scores[:10]:
    print(f"  {sig}...  score={score:.3f}")

# Histogram
fig, ax = plt.subplots(figsize=(10, 4))
all_scores = [s for _, s in scores]
ax.hist(all_scores, bins=50, color='magenta', alpha=0.7, edgecolor='white')
ax.axvline(0.85, color='red', linestyle='--', label='Alert threshold')
ax.set_xlabel('Pump Score')
ax.set_ylabel('Count')
ax.set_title('Heuristic Pump Score Distribution (last 1000 txs)')
ax.legend()
plt.show()

## 7. Summary Stats

In [ ]:
if SUMMARY.exists():
    with open(SUMMARY) as f:
        summary = json.load(f)
    
    print(json.dumps(summary, indent=2, default=str))